In [1]:
from model import alpha, self_energy
from smatrix import create_self_energy_interpolator_numba
import numpy as np
from eigenstate_solving import BZ_proj
from model import square_lattice
import pandas as pd

sigma_data = np.load("../../data/sigma_grid0f1a.npz")
kx = sigma_data["kx"]
ky = sigma_data["ky"]
sigma_grid = sigma_data["sigma_grid"]
sigma_func_period_numba = create_self_energy_interpolator_numba(
    kx, ky, sigma_grid, lattice=square_lattice
)
collective_lamb_shift = self_energy(
    0, 0, square_lattice.a, square_lattice.d, square_lattice.omega_e, alpha
).real
sigma_func_period = create_self_energy_interpolator_numba(
    kx, ky, sigma_grid, lattice=square_lattice
)

E = 2 * (square_lattice.omega_e + collective_lamb_shift) + 0.25

Q_para = np.array([0, 0])
Zc = 0

n_points = int(2e6) # number of grid for FFT

n_r_grid_points = 50 # number of grid points along each direction in the r_para path inside the light cone
r_path = []

vertical_r_grid_points =  np.linspace(0,E/2,n_r_grid_points,endpoint=False)
diagonal_r_grid_points =  np.linspace(0,E/2/np.sqrt(2),n_r_grid_points,endpoint=False)

# go through the grid in reverse order but skip the first point (0,0) to avoid duplication
for x in vertical_r_grid_points[::-1]:
    r_path.append(np.array([x, 0.0]))

#r_path.append(np.array([0.0, 0.0]))
"""
for x in diagonal_r_grid_points[1:]:
    r_path.append(np.array([x, x]))
"""
p_path = r_path.copy()  # use the same path for p_para

eta = 0.0


In [2]:
from W_state import _pole_loc


def state_classifier(r_para, p_para, E1, E, Q_para,eta, lattice, sigma_func_period):
    """Label the state as "bound" or "extended" based on the number of roots of the denominator."""
    if np.linalg.norm(r_para) + np.linalg.norm(BZ_proj(Q_para-r_para,lattice)) > E:
        raise ValueError("In plane-wave basis, the total in-plane photon momenta exceeds the total energy.")

    if np.linalg.norm(p_para) + np.linalg.norm(BZ_proj(Q_para-p_para,lattice)) > E:
        raise ValueError("In W-state label, the total in-plane photon momenta exceeds the total energy.")

    root_list =  _pole_loc(r_para, p_para, E1, E, Q_para,eta, lattice, sigma_func_period)
    q_root_list = sorted(root[0] for root in root_list)
    # If two roots are very close to each other, we treat them as a single double root.
    #if len(q_root_list) == 2 and np.isclose(q_root_list[0], q_root_list[1]):
    #    q_root_list = [(q_root_list[0] + q_root_list[1]) / 2]

    if  len(q_root_list) == 2:
        return "extended"
    elif len(q_root_list) == 0:
        return "bound"
    elif len(q_root_list) == 1:
    #    raise ValueError("Single root found.")
        return "bound"
    else:
        raise ValueError("Unexpected number of roots found: {}".format(len(q_root_list)))

In [3]:
def state_label_dataframe_2D(r_para_values, p_para_values,E1,lattice):
    rows = []
    for p_val in p_para_values:
        p_arr = np.asarray(p_val, dtype=float)
        for r_val in r_para_values:
            r_arr = np.asarray(r_val, dtype=float)
            
            rows.append(
                {   "r_para": tuple(r_arr),
                    "p_para": tuple(p_arr),
                    "r_x": r_arr[0],
                    "p_x": p_arr[0],
                    "E1": E1,
                    "label": state_classifier(
                        r_arr,
                        p_arr,
                        E1,
                        E,
                        Q_para,
                        eta,
                        square_lattice,
                        sigma_func_period,
                    ),
                }
            )
    return pd.DataFrame(rows, columns=["r_para", "p_para", "r_x", "p_x", "E1", "label"])

state_df_2D = state_label_dataframe_2D(r_path, p_path,E/2,square_lattice)
state_df_2D

,r_para,p_para,r_x,p_x,E1,label
0,"(88.10801512151707, 0.0)","(88.10801512151707, 0.0)",88.108015,88.108015,89.906138,bound
1,"(86.30989236393509, 0.0)","(88.10801512151707, 0.0)",86.309892,88.108015,89.906138,extended
2,"(84.5117696063531, 0.0)","(88.10801512151707, 0.0)",84.511770,88.108015,89.906138,extended
3,"(82.71364684877112, 0.0)","(88.10801512151707, 0.0)",82.713647,88.108015,89.906138,extended
4,"(80.91552409118914, 0.0)","(88.10801512151707, 0.0)",80.915524,88.108015,89.906138,extended
...,...,...,...,...,...,...
2495,"(7.192491030327924, 0.0)","(0.0, 0.0)",7.192491,0.000000,89.906138,extended
2496,"(5.394368272745943, 0.0)","(0.0, 0.0)",5.394368,0.000000,89.906138,extended
2497,"(3.596245515163962, 0.0)","(0.0, 0.0)",3.596246,0.000000,89.906138,extended
2498,"(1.798122757581981, 0.0)","(0.0, 0.0)",1.798123,0.000000,89.906138,extended


In [4]:
import plotly.graph_objects as go

fig_2d = go.Figure()

for label, color in [("bound", "blue"), ("extended", "red")]:
    plot_df = state_df_2D.loc[state_df_2D["label"] == label]
    fig_2d.add_trace(
        go.Scatter(
            x=plot_df["r_x"],
            y=plot_df["p_x"],
            mode="markers",
            marker=dict(size=7, color=color, opacity=0.8),
            name=label,
            customdata=plot_df[["E1"]].to_numpy(),
            hovertemplate="r_x=%{x:.3f}<br>p_x=%{y:.3f}<br>E_1=%{customdata[0]:.3f}<extra>%{fullData.name}</extra>",
        )
    )

fig_2d.update_layout(
    title=f"2D state labels at E_1 = {state_df_2D['E1'].iloc[0]:g}",
    xaxis_title="r_x",
    yaxis_title="p_x",
    legend_title_text="state",
    width=750,
    height=650,
)
fig_2d.update_yaxes(scaleanchor="x", scaleratio=1)

fig_2d.show()

In [5]:
import plotly.graph_objects as go


def state_label_dataframe(r_para_values, p_para_values,lattice):
    rows = []
    for p_val in p_para_values:
        p_arr = np.asarray(p_val, dtype=float)
        for r_val in r_para_values:
            r_arr = np.asarray(r_val, dtype=float)
            E1_values = np.linspace(np.linalg.norm(p_arr),E-np.linalg.norm(BZ_proj(Q_para-p_arr,lattice)),10,endpoint=False)

            for E1 in E1_values:
                rows.append(
                    {   "r_para": tuple(r_arr),
                        "p_para": tuple(p_arr),
                        "r_x": r_arr[0],
                        "p_x": p_arr[0],
                        "E1": E1,
                        "label": state_classifier(
                            r_arr,
                            p_arr,
                            E1,
                            E,
                            Q_para,
                            eta,
                            square_lattice,
                            sigma_func_period,
                        ),
                    }
                )
    return pd.DataFrame(rows, columns=["r_para", "p_para", "r_x", "p_x", "E1", "label"])



state_df = state_label_dataframe(r_path, p_path,square_lattice)



fig = go.Figure()

for label, color in [("bound", "blue"), ("extended", "red")]:
    mask = state_df["label"] == label
    fig.add_trace(
        go.Scatter3d(
            x=state_df.loc[mask, "r_x"],
            y=state_df.loc[mask, "p_x"],
            z=state_df.loc[mask, "E1"],
            mode="markers",
            marker=dict(size=3, color=color, opacity=0.7),
            name=label,
        )
    )

fig.update_layout(
    scene=dict(
        xaxis_title="r_x",
        yaxis_title="p_x",
        zaxis_title="E_1",
    ),
    legend_title_text="state",
    width=900,
    height=700,
)

fig.show()
